# TP2 Avancé — Clustering : Méthodes Modernes et Évaluation — Corrigé

Ce corrigé couvre toutes les parties du TP avec des explications détaillées.

In [ ]:
# Installation des dépendances si nécessaire
# !pip install scikit-learn matplotlib numpy pandas umap-learn hdbscan

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Style des graphiques
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## Partie 1 — Rappel et limites de K-Means

In [ ]:
# 1-2. Chargement et K-Means de base
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

url = "https://raw.githubusercontent.com/satishgunjal/datasets/master/Mall_Customers.csv"
df = pd.read_csv(url)
df = df.rename(columns={"Annual Income (k$)": "AnnualIncome", "Spending Score (1-100)": "SpendingScore"})

X_mall = df[["AnnualIncome", "SpendingScore"]].values
scaler = StandardScaler()
X_mall_scaled = scaler.fit_transform(X_mall)

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
labels_kmeans = kmeans.fit_predict(X_mall_scaled)

plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_mall[:, 0], X_mall[:, 1], c=labels_kmeans, cmap='viridis', s=50)
plt.scatter(scaler.inverse_transform(kmeans.cluster_centers_)[:, 0], 
            scaler.inverse_transform(kmeans.cluster_centers_)[:, 1], 
            c='red', marker='X', s=200, label='Centroïdes')
plt.xlabel('Revenu annuel (k$)')
plt.ylabel('Score de dépenses')
plt.title('K-Means sur Mall Customers (k=5)')
plt.legend()
plt.colorbar(scatter, label='Cluster')
plt.tight_layout()
plt.show()

In [ ]:
# 3. Limites de K-Means
print("=== Limites de K-Means ===")
print()
print("1. Sensibilité à l'initialisation")
print("   → Résultats différents selon les centroïdes initiaux")
print("   → Solution : k-means++ (par défaut dans scikit-learn)")
print()
print("2. Hypothèse de clusters sphériques")
print("   → Échoue sur des formes complexes (croissants, anneaux)")
print()
print("3. Nécessité de spécifier k à l'avance")
print("   → Pas toujours évident de choisir le bon nombre")
print()
print("4. Sensibilité aux outliers")
print("   → Les points extrêmes tirent les centroïdes")

In [ ]:
# 4-5. Dataset synthétique : make_moons
from sklearn.datasets import make_moons, make_blobs, make_circles

X_moons, y_moons = make_moons(n_samples=500, noise=0.1, random_state=42)
X_circles, y_circles = make_circles(n_samples=500, noise=0.05, factor=0.5, random_state=42)

# K-Means sur moons
kmeans_moons = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_kmeans_moons = kmeans_moons.fit_predict(X_moons)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='viridis', s=30)
axes[0].set_title('Moons : Labels réels')

axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=labels_kmeans_moons, cmap='viridis', s=30)
axes[1].scatter(kmeans_moons.cluster_centers_[:, 0], kmeans_moons.cluster_centers_[:, 1], 
                c='red', marker='X', s=200)
axes[1].set_title('Moons : K-Means ÉCHOUE')

axes[2].scatter(X_circles[:, 0], X_circles[:, 1], c=y_circles, cmap='viridis', s=30)
axes[2].set_title('Circles : autre cas difficile')

plt.tight_layout()
plt.show()

print("K-Means échoue car il suppose des clusters sphériques !")

## Partie 2 — Clustering Hiérarchique

In [ ]:
# 2.1 Agglomératif
from sklearn.cluster import AgglomerativeClustering

linkages = ['ward', 'complete', 'average', 'single']

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, linkage in zip(axes.flat, linkages):
    agg = AgglomerativeClustering(n_clusters=5, linkage=linkage)
    labels = agg.fit_predict(X_mall_scaled)
    ax.scatter(X_mall[:, 0], X_mall[:, 1], c=labels, cmap='viridis', s=30)
    ax.set_title(f'Linkage: {linkage}')
    ax.set_xlabel('Revenu annuel')
    ax.set_ylabel('Score dépenses')

plt.suptitle('Clustering hiérarchique — Comparaison des linkages', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 2.2 Dendrogramme
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

# Calcul du linkage (sur un échantillon pour lisibilité)
Z = linkage(X_mall_scaled, method='ward')

plt.figure(figsize=(14, 6))
dendrogram(Z, truncate_mode='lastp', p=30, leaf_rotation=90, 
           leaf_font_size=10, show_contracted=True)
plt.title('Dendrogramme (méthode Ward)')
plt.xlabel('Échantillons (agrégés)')
plt.ylabel('Distance')
plt.axhline(y=10, color='r', linestyle='--', label='Coupe à distance=10')
plt.legend()
plt.tight_layout()
plt.show()

# Couper à une certaine distance
clusters_dendro = fcluster(Z, t=10, criterion='distance')
print(f"Nombre de clusters avec distance=10 : {len(np.unique(clusters_dendro))}")

## Partie 3 — DBSCAN

In [ ]:
# 3.2 Application sur make_moons
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(eps=0.2, min_samples=5)
labels_dbscan = dbscan.fit_predict(X_moons)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=labels_kmeans_moons, cmap='viridis', s=30)
axes[0].set_title('K-Means (ÉCHEC)')

axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=labels_dbscan, cmap='viridis', s=30)
axes[1].set_title('DBSCAN (SUCCÈS)')

plt.suptitle('Comparaison K-Means vs DBSCAN sur make_moons', fontsize=14)
plt.tight_layout()
plt.show()

n_outliers = np.sum(labels_dbscan == -1)
print(f"Outliers détectés par DBSCAN : {n_outliers}")

In [ ]:
# 3.3 Choix de eps avec k-distances
from sklearn.neighbors import NearestNeighbors

neighbors = NearestNeighbors(n_neighbors=5)
neighbors.fit(X_mall_scaled)
distances, _ = neighbors.kneighbors(X_mall_scaled)
distances = np.sort(distances[:, -1])

plt.figure(figsize=(10, 5))
plt.plot(distances)
plt.xlabel('Points triés')
plt.ylabel('Distance au 5ème voisin')
plt.title('Graphe des k-distances pour estimer eps')
plt.axhline(y=0.5, color='r', linestyle='--', label='eps ≈ 0.5')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 3.4 DBSCAN sur Mall Customers
dbscan_mall = DBSCAN(eps=0.5, min_samples=5)
labels_dbscan_mall = dbscan_mall.fit_predict(X_mall_scaled)

plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_mall[:, 0], X_mall[:, 1], c=labels_dbscan_mall, cmap='viridis', s=50)

# Marquer les outliers
outliers = labels_dbscan_mall == -1
plt.scatter(X_mall[outliers, 0], X_mall[outliers, 1], c='red', marker='x', s=100, label='Outliers')

plt.xlabel('Revenu annuel (k$)')
plt.ylabel('Score de dépenses')
plt.title(f'DBSCAN sur Mall Customers (eps=0.5, {np.sum(outliers)} outliers)')
plt.legend()
plt.colorbar(scatter, label='Cluster')
plt.tight_layout()
plt.show()

## Partie 4 — HDBSCAN

In [ ]:
import hdbscan

# 4.2 Application
clusterer = hdbscan.HDBSCAN(min_cluster_size=10, min_samples=5)
labels_hdbscan = clusterer.fit_predict(X_mall_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Clusters
scatter = axes[0].scatter(X_mall[:, 0], X_mall[:, 1], c=labels_hdbscan, cmap='viridis', s=50)
axes[0].set_title(f'HDBSCAN ({len(np.unique(labels_hdbscan[labels_hdbscan >= 0]))} clusters)')
axes[0].set_xlabel('Revenu annuel')
axes[0].set_ylabel('Score dépenses')
plt.colorbar(scatter, ax=axes[0])

# 4.3 Probabilités d'appartenance
scatter2 = axes[1].scatter(X_mall[:, 0], X_mall[:, 1], c=clusterer.probabilities_, cmap='RdYlGn', s=50)
axes[1].set_title('Probabilités d\'appartenance')
axes[1].set_xlabel('Revenu annuel')
axes[1].set_ylabel('Score dépenses')
plt.colorbar(scatter2, ax=axes[1], label='Probabilité')

plt.tight_layout()
plt.show()

print(f"Outliers HDBSCAN : {np.sum(labels_hdbscan == -1)}")

## Partie 5 — Gaussian Mixture Models

In [ ]:
# 5.2 Application
from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(n_components=5, random_state=42)
gmm.fit(X_mall_scaled)
labels_gmm = gmm.predict(X_mall_scaled)
probas = gmm.predict_proba(X_mall_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Clusters
axes[0].scatter(X_mall[:, 0], X_mall[:, 1], c=labels_gmm, cmap='viridis', s=50)
axes[0].set_title('GMM Clustering (k=5)')
axes[0].set_xlabel('Revenu annuel')
axes[0].set_ylabel('Score dépenses')

# Probabilité maximale (confiance)
max_proba = probas.max(axis=1)
scatter = axes[1].scatter(X_mall[:, 0], X_mall[:, 1], c=max_proba, cmap='RdYlGn', s=50)
axes[1].set_title('Confiance d\'appartenance (probabilité max)')
axes[1].set_xlabel('Revenu annuel')
plt.colorbar(scatter, ax=axes[1], label='Probabilité')

plt.tight_layout()
plt.show()

In [ ]:
# 5.3 Sélection du nombre de composantes (BIC/AIC)
n_components_range = range(1, 11)
bics = []
aics = []

for k in n_components_range:
    gmm_temp = GaussianMixture(n_components=k, random_state=42)
    gmm_temp.fit(X_mall_scaled)
    bics.append(gmm_temp.bic(X_mall_scaled))
    aics.append(gmm_temp.aic(X_mall_scaled))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(n_components_range, bics, 'b-o', label='BIC')
ax.plot(n_components_range, aics, 'r-s', label='AIC')
ax.axvline(x=np.argmin(bics)+1, color='b', linestyle='--', alpha=0.5)
ax.axvline(x=np.argmin(aics)+1, color='r', linestyle='--', alpha=0.5)
ax.set_xlabel('Nombre de composantes')
ax.set_ylabel('Score')
ax.set_title('Sélection du nombre de composantes GMM')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Optimal selon BIC : {np.argmin(bics)+1} composantes")
print(f"Optimal selon AIC : {np.argmin(aics)+1} composantes")

## Partie 6 — Métriques d'Évaluation

In [ ]:
# 6.1 Métriques internes
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

k_range = range(2, 11)
silhouettes = []
davies_bouldins = []
calinski_harabaszs = []
inertias = []

for k in k_range:
    kmeans_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_temp = kmeans_temp.fit_predict(X_mall_scaled)
    
    silhouettes.append(silhouette_score(X_mall_scaled, labels_temp))
    davies_bouldins.append(davies_bouldin_score(X_mall_scaled, labels_temp))
    calinski_harabaszs.append(calinski_harabasz_score(X_mall_scaled, labels_temp))
    inertias.append(kmeans_temp.inertia_)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].plot(k_range, inertias, 'b-o')
axes[0, 0].set_title('Méthode du coude (Inertie)')
axes[0, 0].set_xlabel('k')
axes[0, 0].set_ylabel('Inertie')

axes[0, 1].plot(k_range, silhouettes, 'g-o')
axes[0, 1].set_title('Silhouette Score (↑ mieux)')
axes[0, 1].set_xlabel('k')
axes[0, 1].axvline(x=k_range[np.argmax(silhouettes)], color='g', linestyle='--')

axes[1, 0].plot(k_range, davies_bouldins, 'r-o')
axes[1, 0].set_title('Davies-Bouldin Index (↓ mieux)')
axes[1, 0].set_xlabel('k')
axes[1, 0].axvline(x=k_range[np.argmin(davies_bouldins)], color='r', linestyle='--')

axes[1, 1].plot(k_range, calinski_harabaszs, 'm-o')
axes[1, 1].set_title('Calinski-Harabasz Index (↑ mieux)')
axes[1, 1].set_xlabel('k')
axes[1, 1].axvline(x=k_range[np.argmax(calinski_harabaszs)], color='m', linestyle='--')

plt.tight_layout()
plt.show()

print(f"Optimal selon Silhouette : k = {k_range[np.argmax(silhouettes)]}")
print(f"Optimal selon Davies-Bouldin : k = {k_range[np.argmin(davies_bouldins)]}")
print(f"Optimal selon Calinski-Harabasz : k = {k_range[np.argmax(calinski_harabaszs)]}")

In [ ]:
# 6.2 Métriques externes (sur Iris)
from sklearn.datasets import load_iris
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics import homogeneity_score, completeness_score, v_measure_score

iris = load_iris()
X_iris = iris.data
y_iris = iris.target

X_iris_scaled = StandardScaler().fit_transform(X_iris)

# Comparaison des algorithmes
algorithms = {
    'K-Means': KMeans(n_clusters=3, random_state=42, n_init=10),
    'DBSCAN': DBSCAN(eps=0.5, min_samples=5),
    'GMM': GaussianMixture(n_components=3, random_state=42)
}

results = []
for name, algo in algorithms.items():
    if name == 'GMM':
        algo.fit(X_iris_scaled)
        labels = algo.predict(X_iris_scaled)
    else:
        labels = algo.fit_predict(X_iris_scaled)
    
    # Filtrer les outliers pour DBSCAN
    mask = labels >= 0
    if mask.sum() < len(labels):
        labels_clean = labels[mask]
        y_clean = y_iris[mask]
    else:
        labels_clean = labels
        y_clean = y_iris
    
    results.append({
        'Algorithme': name,
        'ARI': adjusted_rand_score(y_clean, labels_clean),
        'NMI': normalized_mutual_info_score(y_clean, labels_clean),
        'Homogeneity': homogeneity_score(y_clean, labels_clean),
        'Completeness': completeness_score(y_clean, labels_clean),
        'V-measure': v_measure_score(y_clean, labels_clean)
    })

results_df = pd.DataFrame(results).set_index('Algorithme')
print("=== Métriques externes sur Iris ===")
display(results_df.round(3))

In [ ]:
# 6.3 Analyse de la silhouette par échantillon
from sklearn.metrics import silhouette_samples

labels_k5 = KMeans(n_clusters=5, random_state=42, n_init=10).fit_predict(X_mall_scaled)
sample_silhouette = silhouette_samples(X_mall_scaled, labels_k5)

# Points mal classés (silhouette négative)
mal_classes = np.sum(sample_silhouette < 0)
print(f"Points avec silhouette négative : {mal_classes} ({mal_classes/len(sample_silhouette)*100:.1f}%)")

# Visualisation
plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_mall[:, 0], X_mall[:, 1], c=sample_silhouette, cmap='RdYlGn', 
                      vmin=-1, vmax=1, s=50)
plt.colorbar(scatter, label='Silhouette')
plt.xlabel('Revenu annuel')
plt.ylabel('Score dépenses')
plt.title('Silhouette par échantillon (rouge = mal classé)')
plt.tight_layout()
plt.show()

## Partie 7 — Réduction de Dimension

In [ ]:
# Dataset plus complexe pour la visualisation
from sklearn.datasets import load_digits

digits = load_digits()
X_digits = digits.data
y_digits = digits.target

print(f"Dimensions des digits : {X_digits.shape} (64 features = 8x8 pixels)")

In [ ]:
# 7.1-7.3 PCA, t-SNE, UMAP
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap

# PCA
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_digits)

# t-SNE
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_tsne = tsne.fit_transform(X_digits)

# UMAP
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
X_umap = reducer.fit_transform(X_digits)

# 7.4 Comparaison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (X_reduced, title) in zip(axes, [(X_pca, 'PCA'), (X_tsne, 't-SNE'), (X_umap, 'UMAP')]):
    scatter = ax.scatter(X_reduced[:, 0], X_reduced[:, 1], c=y_digits, cmap='tab10', s=10, alpha=0.7)
    ax.set_title(title, fontsize=14)
    ax.set_xticks([])
    ax.set_yticks([])

plt.colorbar(scatter, ax=axes, label='Chiffre', shrink=0.8)
plt.suptitle('Comparaison des méthodes de réduction de dimension (Digits)', fontsize=14)
plt.tight_layout()
plt.show()

print("t-SNE et UMAP séparent mieux les clusters que PCA !")

## Partie 8 — Cas d'étude complet

In [ ]:
# Dataset Wholesale Customers
url_wholesale = "https://archive.ics.uci.edu/ml/machine-learning-databases/00292/Wholesale%20customers%20data.csv"
df_wholesale = pd.read_csv(url_wholesale)

print("=== Exploration ===")
display(df_wholesale.head())
print(f"\nShape: {df_wholesale.shape}")
print(f"\nStatistiques:")
display(df_wholesale.describe())

In [ ]:
# Prétraitement
# On garde les colonnes numériques de dépenses
features = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
X_wholesale = df_wholesale[features].values

# Normalisation
scaler_wholesale = StandardScaler()
X_wholesale_scaled = scaler_wholesale.fit_transform(X_wholesale)

print(f"Features utilisées : {features}")

In [ ]:
# Comparaison des algorithmes
algorithms_wholesale = {
    'K-Means (k=3)': KMeans(n_clusters=3, random_state=42, n_init=10),
    'K-Means (k=4)': KMeans(n_clusters=4, random_state=42, n_init=10),
    'Hiérarchique (k=3)': AgglomerativeClustering(n_clusters=3, linkage='ward'),
    'DBSCAN': DBSCAN(eps=1.5, min_samples=5),
    'GMM (k=3)': GaussianMixture(n_components=3, random_state=42)
}

results_wholesale = []
all_labels = {}

for name, algo in algorithms_wholesale.items():
    if 'GMM' in name:
        algo.fit(X_wholesale_scaled)
        labels = algo.predict(X_wholesale_scaled)
    else:
        labels = algo.fit_predict(X_wholesale_scaled)
    
    all_labels[name] = labels
    
    # Filtrer outliers pour métriques
    mask = labels >= 0
    if mask.sum() >= 2 and len(np.unique(labels[mask])) >= 2:
        sil = silhouette_score(X_wholesale_scaled[mask], labels[mask])
        db = davies_bouldin_score(X_wholesale_scaled[mask], labels[mask])
    else:
        sil, db = np.nan, np.nan
    
    n_clusters = len(np.unique(labels[labels >= 0]))
    n_outliers = np.sum(labels == -1)
    
    results_wholesale.append({
        'Algorithme': name,
        'N clusters': n_clusters,
        'Outliers': n_outliers,
        'Silhouette': sil,
        'Davies-Bouldin': db
    })

results_df_wholesale = pd.DataFrame(results_wholesale).set_index('Algorithme')
print("=== Comparaison des algorithmes ===")
display(results_df_wholesale.round(3))

In [ ]:
# Visualisation UMAP avec le meilleur algorithme
best_algo = results_df_wholesale['Silhouette'].idxmax()
best_labels = all_labels[best_algo]

reducer_wholesale = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
X_wholesale_umap = reducer_wholesale.fit_transform(X_wholesale_scaled)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_wholesale_umap[:, 0], X_wholesale_umap[:, 1], 
                      c=best_labels, cmap='viridis', s=50, alpha=0.7)
plt.colorbar(scatter, label='Cluster')
plt.title(f'Visualisation UMAP — {best_algo}')
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.tight_layout()
plt.show()

In [ ]:
# Interprétation : profil moyen par cluster
df_wholesale['Cluster'] = best_labels

print(f"=== Profil moyen par cluster ({best_algo}) ===")
profile = df_wholesale.groupby('Cluster')[features].mean()
display(profile.round(0))

# Visualisation radar/heatmap
profile_normalized = (profile - profile.min()) / (profile.max() - profile.min())

plt.figure(figsize=(12, 5))
im = plt.imshow(profile_normalized.T, cmap='YlOrRd', aspect='auto')
plt.colorbar(im, label='Valeur normalisée')
plt.yticks(range(len(features)), features)
plt.xticks(range(len(profile)), [f'Cluster {i}' for i in profile.index])
plt.title('Profil des clusters (valeurs normalisées)')
plt.tight_layout()
plt.show()

## Partie 9 — Exercices bonus

In [ ]:
# Bonus 1 : Spectral Clustering
from sklearn.cluster import SpectralClustering

spectral = SpectralClustering(n_clusters=2, affinity='nearest_neighbors', random_state=42)
labels_spectral = spectral.fit_predict(X_moons)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='viridis', s=30)
axes[0].set_title('Labels réels')

axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=labels_dbscan, cmap='viridis', s=30)
axes[1].set_title('DBSCAN')

axes[2].scatter(X_moons[:, 0], X_moons[:, 1], c=labels_spectral, cmap='viridis', s=30)
axes[2].set_title('Spectral Clustering')

plt.suptitle('Spectral Clustering sur make_moons')
plt.tight_layout()
plt.show()

In [ ]:
# Bonus 2 : Mini-Batch K-Means
from sklearn.cluster import MiniBatchKMeans
import time

# Grand dataset synthétique
X_large, _ = make_blobs(n_samples=50000, n_features=10, centers=5, random_state=42)

# K-Means classique
start = time.time()
kmeans_classic = KMeans(n_clusters=5, random_state=42, n_init=10)
kmeans_classic.fit(X_large)
time_classic = time.time() - start

# Mini-Batch K-Means
start = time.time()
kmeans_mini = MiniBatchKMeans(n_clusters=5, random_state=42, batch_size=1000)
kmeans_mini.fit(X_large)
time_mini = time.time() - start

print("=== Comparaison K-Means vs Mini-Batch K-Means ===")
print(f"K-Means classique : {time_classic:.2f}s, inertie = {kmeans_classic.inertia_:.0f}")
print(f"Mini-Batch K-Means : {time_mini:.2f}s, inertie = {kmeans_mini.inertia_:.0f}")
print(f"\nAccélération : {time_classic/time_mini:.1f}x plus rapide")